In [ ]:
def main(datasources, start_date, end_date):
    """
    单因子：Residual Target ML Distillation Factor

    目标：
        专门冲 B 项稳定性。

    核心：
        1. 使用 t 日收盘后可得信息；
        2. 预测 t+1 日 open-to-close 收益：
              label_t = close_{t+1} / open_{t+1} - 1
        3. 对 label 做每日横截面 z-score；
        4. 用 bigalpha_2026_exposure 对 label_z 做风格/行业残差化；
        5. 用日内结构 + 流动性 + 反转 + 承接类 rank 子因子拟合 residual target；
        6. 测试期输出模型预测值的每日 rank。

    输出：
        ['date', 'instrument', 'factor']
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog

    logger = structlog.get_logger()
    EPS = 1e-12

    TRAIN_START = '2019-01-01 00:00:00'
    TRAIN_END = '2024-12-31 23:59:59'

    # 是否使用 XGBoost。若环境没有 xgboost，则自动退回 Ridge。
    USE_XGB = True

    # ============================================================
    # 风格与行业暴露列：与评估函数加载的 bigalpha_2026_exposure 对齐
    # ============================================================
    EXPOSURE_COLS = [
        'SIZE', 'BETA', 'MOMENTUM', 'RESVOL', 'SIZENL', 'BTOP',
        'LIQUIDTY', 'EARNYILD', 'GROWTH', 'LEVERAGE',
        'AGRIFOREST', 'MINING', 'CHEM', 'IRONSTEEL', 'NONFERMETAL',
        'ELECTRONICS', 'AUTO', 'HOUSEAPP', 'FOODBEVER', 'TEXTILE',
        'LIGHTINDUS', 'HEALTH', 'UTILITIES', 'TRANSPORTATION',
        'REALESTATE', 'COMMETRADE', 'LEISERVICE', 'BANK',
        'NONBANKFINAN', 'CONGLOMERATES', 'CONMAT', 'BUILDDECO',
        'ELECEQP', 'AERODEF', 'COMPUTER', 'MEDIA', 'TELECOM',
        'COAL', 'PETRO', 'ENVP', 'BEAUTY'
    ]

    # ============================================================
    # 横截面工具
    # ============================================================
    def cs_winsorize(s, q=0.01):
        s = s.replace([np.inf, -np.inf], np.nan)
        if s.notna().sum() < 20:
            return s
        lo = s.quantile(q)
        hi = s.quantile(1 - q)
        return s.clip(lo, hi)

    def cs_zscore(s):
        s = s.replace([np.inf, -np.inf], np.nan)
        mu = s.mean()
        sd = s.std()
        return (s - mu) / (sd + EPS)

    def cs_rank(s):
        s = s.replace([np.inf, -np.inf], np.nan)
        return s.rank(pct=True) - 0.5

    def add_cs_z(df, col, out_col):
        df[out_col] = (
            df.groupby('date')[col]
              .transform(lambda x: cs_zscore(cs_winsorize(x, q=0.01)))
        )
        return df

    def add_cs_rank(df, col, out_col):
        df[out_col] = (
            df.groupby('date')[col]
              .transform(lambda x: cs_rank(cs_winsorize(x, q=0.01)))
        )
        return df

    # ============================================================
    # 查询风险暴露
    # ============================================================
    def query_exposure(sd, ed):
        sql = f"""
        SELECT date, instrument, {', '.join(EXPOSURE_COLS)}
        FROM bigalpha_2026_exposure
        """
        expo = dai.query(sql, filters={'date': [sd, ed]}).df()
        expo['date'] = pd.to_datetime(expo['date'])
        expo['instrument'] = expo['instrument'].astype(str)

        for c in EXPOSURE_COLS:
            if c in expo.columns:
                expo[c] = pd.to_numeric(expo[c], errors='coerce')
                expo[c] = expo[c].replace([np.inf, -np.inf], np.nan)

        return expo

    # ============================================================
    # 每日横截面残差化
    # ============================================================
    def residualize_by_date(df, y_col, x_cols, out_col):
        out_list = []

        keep_cols = ['date', 'instrument', y_col] + x_cols
        use = df[keep_cols].replace([np.inf, -np.inf], np.nan)

        for dt, sub in use.groupby('date'):
            sub = sub.dropna(subset=[y_col]).copy()

            if len(sub) < max(80, len(x_cols) + 20):
                tmp = sub[['date', 'instrument']].copy()
                tmp[out_col] = sub[y_col] - sub[y_col].mean()
                out_list.append(tmp)
                continue

            X_df = sub[x_cols].copy()

            # 暴露缺失用当日中位数填充
            for c in x_cols:
                med = X_df[c].median()
                X_df[c] = X_df[c].fillna(0.0 if pd.isna(med) else med)

            # 标准化 X，避免尺度病态
            X = X_df.values.astype(float)
            X_mu = np.nanmean(X, axis=0)
            X_sd = np.nanstd(X, axis=0) + EPS
            X = (X - X_mu) / X_sd

            y = sub[y_col].values.astype(float)

            # 加截距
            X = np.column_stack([np.ones(len(X)), X])

            try:
                beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
                resid = y - X @ beta
            except Exception:
                resid = y - np.nanmean(y)

            tmp = sub[['date', 'instrument']].copy()
            tmp[out_col] = resid
            out_list.append(tmp)

        if len(out_list) == 0:
            df[out_col] = np.nan
            return df

        resid_df = pd.concat(out_list, axis=0)
        df = df.merge(resid_df, on=['date', 'instrument'], how='left')
        return df

    # ============================================================
    # 构造训练/测试共用特征
    # ============================================================
    def build_dataset(bar1m_table, sd, ed, need_label=False):
        t0 = time.time()
        logger.info("build_dataset 开始", start=str(sd), end=str(ed), need_label=need_label)

        price_start = pd.to_datetime(sd) - pd.Timedelta(days=140)

        query_end = pd.to_datetime(ed)
        if need_label:
            query_end = query_end + pd.Timedelta(days=20)

        price_sql = f"""
        WITH minute AS (
            SELECT
                date,
                date_trunc('day', date)::DATE AS trading_day,
                instrument,
                open,
                high,
                low,
                close,
                volume,
                amount,
                close / NULLIF(open, 0) - 1 AS ret_m,
                CASE
                    WHEN high > low THEN (close - open) / NULLIF(high - low, 0)
                    ELSE 0
                END AS clv_m
            FROM {bar1m_table}
        ),
        agg AS (
            SELECT
                trading_day,
                instrument,

                ARG_MIN(open, date) AS open,
                MAX(high) AS high,
                MIN(low) AS low,
                ARG_MAX(close, date) AS close,

                SUM(volume) AS volume,
                SUM(amount) AS amount,

                SUM(amount * clv_m) / NULLIF(SUM(amount), 0) AS signed_flow,

                SUM(
                    CASE
                        WHEN ret_m < 0 THEN amount * ABS(ret_m)
                        ELSE 0
                    END
                ) / NULLIF(SUM(amount), 0) AS downside_pressure,

                SUM(
                    CASE
                        WHEN ret_m > 0 THEN amount * ABS(ret_m)
                        ELSE 0
                    END
                ) / NULLIF(SUM(amount), 0) AS upside_pressure,

                SUM(
                    CASE
                        WHEN ret_m < 0 THEN amount
                        ELSE 0
                    END
                ) / NULLIF(SUM(amount), 0) AS down_amount_share,

                SUM(
                    CASE
                        WHEN ret_m > 0 THEN amount
                        ELSE 0
                    END
                ) / NULLIF(SUM(amount), 0) AS up_amount_share,

                SUM(POWER(ret_m, 2)) AS rv2,
                SUM(POWER(ret_m, 3)) AS rv3,

                AVG(ret_m) AS intra_mean_ret,
                STDDEV_SAMP(ret_m) AS intra_std_ret

            FROM minute
            GROUP BY trading_day, instrument
        )
        SELECT *
        FROM agg
        ORDER BY trading_day, instrument
        """

        price = (
            dai.query(price_sql, filters={'date': [price_start, query_end]})
               .df()
               .rename(columns={'trading_day': 'date'})
        )

        price['date'] = pd.to_datetime(price['date'])
        price['instrument'] = price['instrument'].astype(str)

        num_cols = [
            'open', 'high', 'low', 'close',
            'volume', 'amount',
            'signed_flow',
            'downside_pressure',
            'upside_pressure',
            'down_amount_share',
            'up_amount_share',
            'rv2',
            'rv3',
            'intra_mean_ret',
            'intra_std_ret',
        ]

        for c in num_cols:
            price[c] = pd.to_numeric(price[c], errors='coerce')
            price[c] = price[c].replace([np.inf, -np.inf], np.nan)

        price = price.sort_values(['instrument', 'date']).reset_index(drop=True)

        g = price.groupby('instrument', group_keys=False)

        # ========================================================
        # 标签：下一交易日 open-to-close 收益
        # ========================================================
        price['oc_ret'] = price['close'] / (price['open'] + EPS) - 1

        if need_label:
            price['label'] = g['oc_ret'].shift(-1)
            price['label_z'] = (
                price.groupby('date')['label']
                     .transform(lambda x: cs_zscore(cs_winsorize(x, q=0.01)))
            )

        # ========================================================
        # 日频与日内基础变量
        # ========================================================
        price['ret1'] = g['close'].pct_change()
        price['ret2'] = g['close'].pct_change(2)
        price['ret3'] = g['close'].pct_change(3)
        price['ret5'] = g['close'].pct_change(5)
        price['ret10'] = g['close'].pct_change(10)

        price['mkt_ret'] = price.groupby('date')['ret1'].transform('mean')
        price['res_ret'] = price['ret1'] - price['mkt_ret']

        price['vol5'] = (
            price.groupby('instrument')['ret1']
                 .rolling(5, min_periods=3)
                 .std()
                 .reset_index(level=0, drop=True)
        )

        price['vol10'] = (
            price.groupby('instrument')['ret1']
                 .rolling(10, min_periods=5)
                 .std()
                 .reset_index(level=0, drop=True)
        )

        price['vol20'] = (
            price.groupby('instrument')['ret1']
                 .rolling(20, min_periods=10)
                 .std()
                 .reset_index(level=0, drop=True)
        )

        price['res_vol20'] = (
            price.groupby('instrument')['res_ret']
                 .rolling(20, min_periods=10)
                 .std()
                 .reset_index(level=0, drop=True)
        )

        price['range'] = (price['high'] - price['low']) / (price['close'] + EPS)

        price['close_pos'] = (
            (price['close'] - price['low']) /
            (price['high'] - price['low'] + EPS)
        )

        price['vwap'] = price['amount'] / (price['volume'] + EPS)
        price['vwap_gap'] = price['close'] / (price['vwap'] + EPS) - 1

        price['log_amount'] = np.log1p(price['amount'])

        price['log_amount_ma20'] = (
            price.groupby('instrument')['log_amount']
                 .rolling(20, min_periods=10)
                 .mean()
                 .reset_index(level=0, drop=True)
        )

        price['amount_shock20'] = price['log_amount'] - price['log_amount_ma20']

        price['amihud'] = np.abs(price['ret1']) / (price['amount'] + EPS)

        price['intra_skew'] = price['rv3'] / (np.power(price['rv2'], 1.5) + EPS)
        price['rv'] = np.sqrt(price['rv2'].clip(lower=0))

        # ========================================================
        # 横截面 z-score
        # ========================================================
        z_cols = [
            'ret1', 'ret2', 'ret3', 'ret5', 'ret10',
            'res_ret',
            'vol5', 'vol10', 'vol20', 'res_vol20',
            'range',
            'close_pos',
            'vwap_gap',
            'log_amount',
            'amount_shock20',
            'signed_flow',
            'downside_pressure',
            'upside_pressure',
            'down_amount_share',
            'up_amount_share',
            'amihud',
            'intra_skew',
            'rv',
            'intra_mean_ret',
            'intra_std_ret',
        ]

        for c in z_cols:
            price = add_cs_z(price, c, 'z_' + c)

        price['amount_shock_pos'] = price['z_amount_shock20'].clip(lower=0)

        # ========================================================
        # 原始子信号
        # ========================================================

        # 1. 市场中性残差反转
        price['raw_idio_rev1'] = -price['res_ret'] / (price['res_vol20'] + EPS)

        # 2. 多周期波动率调整反转
        price['raw_rev2'] = -price['ret2'] / (price['vol20'] + EPS)
        price['raw_rev3'] = -price['ret3'] / (price['vol20'] + EPS)
        price['raw_rev5'] = -price['ret5'] / (price['vol20'] + EPS)
        price['raw_rev10'] = -price['ret10'] / (price['vol20'] + EPS)

        # 3. 放量下跌修复
        price['down_ret'] = (-price['ret1']).clip(lower=0)
        price['raw_down_volume_repair'] = (
            price['down_ret'] / (price['vol20'] + EPS)
            * price['amount_shock_pos']
        )

        # 4. 放量上涨回撤
        price['up_ret'] = price['ret1'].clip(lower=0)
        price['raw_up_volume_reversal'] = (
            -price['up_ret'] / (price['vol20'] + EPS)
            * price['amount_shock_pos']
        )

        # 5. 卖压承接
        price['raw_absorption'] = (
            (-price['z_signed_flow']).clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 6. 下行冲击修复
        price['raw_downside_repair'] = (
            price['z_downside_pressure'].clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 7. VWAP 修复
        price['raw_vwap_reclaim'] = (
            price['z_vwap_gap']
            * (-price['z_signed_flow']).clip(lower=0)
        )

        # 8. 日内强势延续
        price['raw_vwap_strength'] = (
            price['z_vwap_gap'].clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 9. 低波动防御
        price['raw_lowvol'] = -price['vol20']

        # 10. 非流动性修复
        price['raw_amihud_repair'] = (
            price['z_amihud'].clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 11. 日内负偏修复
        price['raw_skew_repair'] = (
            -price['z_intra_skew']
            * price['z_close_pos'].clip(lower=0)
        )

        # 12. 卖压耗尽
        price['raw_selloff_exhaustion'] = (
            price['z_downside_pressure'].clip(lower=0)
            * (-price['z_signed_flow']).clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 13. 下跌成交占比修复
        price['raw_down_share_repair'] = (
            price['z_down_amount_share'].clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 14. 上涨成交占比回撤
        price['raw_up_share_reversal'] = (
            -price['z_up_amount_share'].clip(lower=0)
            * price['z_close_pos'].clip(lower=0)
        )

        # 15. 日内 realized vol 压缩
        price['raw_rv_defensive'] = -price['z_rv']

        # 16. 日内均值反转
        price['raw_intraday_mean_rev'] = -price['z_intra_mean_ret']

        raw_factor_cols = [
            'raw_idio_rev1',
            'raw_rev2',
            'raw_rev3',
            'raw_rev5',
            'raw_rev10',
            'raw_down_volume_repair',
            'raw_up_volume_reversal',
            'raw_absorption',
            'raw_downside_repair',
            'raw_vwap_reclaim',
            'raw_vwap_strength',
            'raw_lowvol',
            'raw_amihud_repair',
            'raw_skew_repair',
            'raw_selloff_exhaustion',
            'raw_down_share_repair',
            'raw_up_share_reversal',
            'raw_rv_defensive',
            'raw_intraday_mean_rev',
        ]

        factor_cols = []

        for raw_col in raw_factor_cols:
            fac_col = raw_col.replace('raw_', 'fac_')
            price = add_cs_rank(price, raw_col, fac_col)
            price[fac_col] = price[fac_col].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            factor_cols.append(fac_col)

        # 只保留目标区间
        price = price[
            (price['date'] >= pd.to_datetime(sd)) &
            (price['date'] <= pd.to_datetime(ed))
        ].copy()

        keep_cols = ['date', 'instrument'] + factor_cols
        if need_label:
            keep_cols += ['label', 'label_z']

        out = price[keep_cols].copy()

        logger.info(
            "build_dataset 结束",
            rows=len(out),
            total_elapsed=round(time.time() - t0, 2)
        )

        return out.reset_index(drop=True), factor_cols

    # ============================================================
    # 训练 XGBoost 残差目标模型
    # ============================================================
    def fit_models(train_df, factor_cols):
        # 合并训练期 exposure，并构造残差目标
        expo = query_exposure(TRAIN_START, TRAIN_END)

        data = train_df.merge(expo, how='left', on=['date', 'instrument'])

        data = residualize_by_date(
            data,
            y_col='label_z',
            x_cols=EXPOSURE_COLS,
            out_col='target_resid'
        )

        use = (
            data[['date', 'instrument', 'target_resid'] + factor_cols]
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=['target_resid'])
            .copy()
        )

        for c in factor_cols:
            use[c] = use[c].fillna(0.0)

        X = use[factor_cols].values.astype(float)
        y = use['target_resid'].values.astype(float)
        y = y - np.nanmean(y)

        models = []

        if USE_XGB:
            try:
                import xgboost as xgb

                # 4 个浅层强正则模型，降低单模型偶然性
                params_list = [
                    dict(n_estimators=60, max_depth=2, learning_rate=0.035,
                         subsample=0.75, colsample_bytree=0.80,
                         min_child_weight=50, reg_lambda=20.0, reg_alpha=0.1,
                         tree_method='hist', n_jobs=-1, random_state=11),
                    dict(n_estimators=80, max_depth=2, learning_rate=0.030,
                         subsample=0.70, colsample_bytree=0.75,
                         min_child_weight=80, reg_lambda=30.0, reg_alpha=0.2,
                         tree_method='hist', n_jobs=-1, random_state=22),
                    dict(n_estimators=50, max_depth=3, learning_rate=0.025,
                         subsample=0.70, colsample_bytree=0.70,
                         min_child_weight=100, reg_lambda=40.0, reg_alpha=0.3,
                         tree_method='hist', n_jobs=-1, random_state=33),
                ]

                for p in params_list:
                    model = xgb.XGBRegressor(**p)
                    model.fit(X, y)
                    models.append(('xgb', model))

                logger.info("XGBoost 残差模型训练完成", n_models=len(models))

            except Exception as e:
                logger.warning("XGBoost 不可用，回退到 Ridge", error=str(e))

        # 如果 XGB 失败或禁用，使用 Ridge 兜底
        if len(models) == 0:
            alpha = 5000.0
            k = X.shape[1]
            beta = np.linalg.solve(X.T @ X + alpha * np.eye(k), X.T @ y)
            models.append(('ridge', beta))
            logger.info("Ridge 残差模型训练完成")

        return models

    def predict_models(models, X):
        preds = []

        for kind, model in models:
            if kind == 'xgb':
                preds.append(model.predict(X))
            elif kind == 'ridge':
                preds.append(X @ model)

        return np.mean(np.column_stack(preds), axis=1)

    # ============================================================
    # 第 1 步：训练
    # ============================================================
    logger.info("开始构造训练集", train_start=TRAIN_START, train_end=TRAIN_END)

    train_df, factor_cols = build_dataset(
        'bigalpha_2026_stock_bar1m',
        TRAIN_START,
        TRAIN_END,
        need_label=True
    )

    models = fit_models(train_df, factor_cols)

    # ============================================================
    # 第 2 步：测试期预测
    # ============================================================
    logger.info("开始构造测试集并预测", start=start_date, end=end_date)

    test_df, _ = build_dataset(
        datasources['bar1m'],
        start_date,
        end_date,
        need_label=False
    )

    for c in factor_cols:
        if c not in test_df.columns:
            test_df[c] = 0.0
        test_df[c] = test_df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    X_test = test_df[factor_cols].values.astype(float)

    test_df['raw_pred'] = predict_models(models, X_test)

    # ============================================================
    # 第 3 步：对预测值再做一次 exposure + 常见代理变量残差化
    # 目的：减少和普通因子/风格因子的重叠，提高 B 项稳定性
    # ============================================================
    expo_test = query_exposure(start_date, end_date)
    test_df = test_df.merge(expo_test, how='left', on=['date', 'instrument'])

    common_proxy_cols = [
        'fac_idio_rev1',
        'fac_rev5',
        'fac_down_volume_repair',
        'fac_lowvol',
        'fac_vwap_strength',
    ]

    resid_cols = EXPOSURE_COLS + [c for c in common_proxy_cols if c in test_df.columns]

    test_df = residualize_by_date(
        test_df,
        y_col='raw_pred',
        x_cols=resid_cols,
        out_col='raw_pred_resid'
    )

    # 最终每日 rank 输出
    test_df['factor'] = (
        test_df.groupby('date')['raw_pred_resid']
               .transform(lambda x: cs_rank(cs_winsorize(x, q=0.01)))
    )

    test_df['factor'] = test_df['factor'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    factor_df = test_df[['date', 'instrument', 'factor']].copy()

    # ============================================================
    # 第 4 步：对齐中证 1000 股票池
    # ============================================================
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()

    stk_pool['date'] = pd.to_datetime(stk_pool['date'])
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    result = pd.merge(
        factor_df,
        stk_pool,
        how='inner',
        on=['date', 'instrument']
    )

    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)

    result = (
        result.dropna(subset=['factor'])
              .reset_index(drop=True)[['date', 'instrument', 'factor']]
    )

    logger.info("因子构建完成", rows=len(result))
    return result


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }

    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        show=True,
    )